# 00 - 环境配置与连通性验证

本 Notebook 完成以下任务：
1. 安装 invest_model 包（开发模式）
2. 验证 `.env` 环境变量
3. 测试 MySQL 数据库连接
4. 测试 Tushare 数据源连通性
5. 测试 Baostock 数据源连通性

---

## 技术方案

- **配置管理**：`config/config.yaml` 存放非敏感配置，`.env` 存放密钥和数据库连接
- **配置加载**：`invest_model.config` 模块合并 yaml + env，全局缓存
- **数据库**：MySQL (RDS)，通过 SQLAlchemy Engine 连接，连接池复用
- **数据源**：Tushare Pro (主，5000+ 积分) + Baostock (补充，免费)
- **日志**：loguru，控制台 INFO + 文件 DEBUG + 错误单独文件

## 1. 安装 invest_model 包

In [ ]:
# 以开发模式安装（-e），代码修改后无需重新安装
!cd /home/admin/Code/invest-journey/invest-model && pip install -e ".[dev]" -q

## 2. 验证环境变量

`.env` 文件需包含以下变量：
```
TUSHARE_TOKEN=你的token
MYSQL_HOST=数据库地址
MYSQL_PORT=3306
MYSQL_USER=用户名
MYSQL_PASSWORD=密码
MYSQL_DATABASE=invest
```

In [ ]:
from invest_model.config import load_config, get_env, get_mysql_url

cfg = load_config()
print("config.yaml 加载成功")
print(f"  数据源: Tushare={cfg['sources']['tushare']['enabled']}, Baostock={cfg['sources']['baostock']['enabled']}")
print(f"  历史回溯: {cfg['collection']['history_years']} 年")

print(f"\nTUSHARE_TOKEN: {get_env('TUSHARE_TOKEN')[:8]}...（已脱敏）")
print(f"MYSQL_HOST: {get_env('MYSQL_HOST')}")
print(f"MYSQL_DATABASE: {get_env('MYSQL_DATABASE')}")

url = get_mysql_url()
print(f"\nMySQL URL: {url.split('@')[0].split('://')[0]}://***@{url.split('@')[1]}")

## 3. 测试 MySQL 连接

In [ ]:
from invest_model.db import test_connection, get_engine

if test_connection():
    print("MySQL 连接成功")
    engine = get_engine()
    with engine.connect() as conn:
        from sqlalchemy import text
        result = conn.execute(text("SELECT VERSION()"))
        version = result.scalar()
        print(f"MySQL 版本: {version}")
else:
    print("MySQL 连接失败，请检查 .env 中的数据库配置")

## 4. 测试 Tushare 连通性

In [ ]:
from invest_model.sources.tushare_client import TushareClient

ts_client = TushareClient()

# 拉取最近 5 个交易日的日历作为连通性测试
cal = ts_client.get_trade_calendar("20260401", "20260406")
print(f"Tushare 连通性测试通过，获取到 {len(cal)} 条日历记录：")
print(cal.to_string(index=False))

## 5. 测试 Baostock 连通性

In [ ]:
from invest_model.sources.baostock_client import BaostockClient

bao_client = BaostockClient()

# 拉取上证指数最近几天日线
df = bao_client.get_stock_daily("600000.SH", "20260401", "20260406")
print(f"Baostock 连通性测试通过，获取到 {len(df)} 条日线记录：")
print(df.head().to_string(index=False))

## 验证总结

| 项目 | 状态 |
|------|------|
| invest_model 包安装 | 运行上方 cell 确认 |
| .env 环境变量 | 运行上方 cell 确认 |
| MySQL 连接 | 运行上方 cell 确认 |
| Tushare 连通 | 运行上方 cell 确认 |
| Baostock 连通 | 运行上方 cell 确认 |

全部通过后，继续 `01_database.ipynb` 初始化数据库表。